# 🔥 OTAKU UPSCALER V1 — TOPAZ INSPIRED

AI image upscaler for anime and movies.

**Workflow:** Upload → Profile → AI Upscale → Enhance → Preview → Download

In [ ]:
# ============================================
# 🔥 OTAKU UPSCALER V1 — GPU CHECK
# ============================================
!nvidia-smi

print('🔥 OTAKU UPSCALER V1')
print('====================')
print('GPU environment detected.')

In [ ]:
# ============================================
# 🧠 OTAKU AI ENVIRONMENT
# ============================================
!apt-get -qq update
!apt-get -qq install python3.11 python3.11-venv
!rm -rf /content/venv311
!python3.11 -m venv /content/venv311
!/content/venv311/bin/python -m ensurepip --upgrade
!/content/venv311/bin/python -m pip install --upgrade pip
!/content/venv311/bin/pip install -q basicsr facexlib gfpgan
%cd /content
!rm -rf /content/realesrgan
!git clone -q https://github.com/xinntao/Real-ESRGAN.git realesrgan
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /content/venv311/lib/python3.11/site-packages/basicsr/data/degradations.py
print('✅ OTAKU AI environment ready')

In [ ]:
# ============================================
# 🛠️ REAL-ESRGAN COMPATIBILITY
# ============================================
from pathlib import Path
version_file = Path('/content/realesrgan/realesrgan/version.py')
version_file.write_text('__version__ = "0.3.0"\n')
print('✅ Compatibility layer ready')

In [ ]:
# ============================================
# 📦 REAL-ESRGAN MODEL
# ============================================
!mkdir -p /content/realesrgan/weights
!rm -f /content/realesrgan/weights/RealESRGAN_x4plus.pth
!wget -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -O /content/realesrgan/weights/RealESRGAN_x4plus.pth
!ls -lh /content/realesrgan/weights/RealESRGAN_x4plus.pth
print('🔥 Real-ESRGAN model ready')

In [ ]:
# ============================================
# 📁 OTAKU FILE UPLOAD
# ============================================
from google.colab import files
from pathlib import Path

input_folder = Path('/content/input')
output_folder = Path('/content/output')
input_folder.mkdir(parents=True, exist_ok=True)
output_folder.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

for filename, data in uploaded.items():
    (input_folder / filename).write_bytes(data)

print('\n🔥 Uploaded files:')
for filename in uploaded:
    print('✓', filename)

In [ ]:
# ============================================
# ⚡ OTAKU TOPAZ-INSPIRED PROFILE
# ============================================
TOPAZ_PROFILE = {
    'revert_compression': 74,
    'recover_details': 88,
    'sharpen': 40,
    'reduce_noise': 59,
    'dehalo': 13,
    'anti_alias_deblur': -22,
    'output_scale': 2
}

print('🔥 OTAKU UPSCALER V1')
print('====================')
print('Profile: TOPAZ INSPIRED')
print()
print('Revert Compression :', TOPAZ_PROFILE['revert_compression'])
print('Recover Details    :', TOPAZ_PROFILE['recover_details'])
print('Sharpen            :', TOPAZ_PROFILE['sharpen'])
print('Reduce Noise       :', TOPAZ_PROFILE['reduce_noise'])
print('Dehalo             :', TOPAZ_PROFILE['dehalo'])
print('Anti-Alias/DeBlur  :', TOPAZ_PROFILE['anti_alias_deblur'])
print('Output Scale       :', TOPAZ_PROFILE['output_scale'], '×')

In [ ]:
# ============================================
# 🧠 OTAKU AI UPSCALING ENGINE
# ============================================
import subprocess
import cv2
from pathlib import Path
from PIL import Image
from IPython.display import display

image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
input_files = [p for p in input_folder.iterdir() if p.suffix.lower() in image_extensions]

if not input_files:
    raise Exception('❌ No supported image found. Run the upload cell first.')

input_file = input_files[0]
scale = TOPAZ_PROFILE['output_scale']

print('🔥 OTAKU AI ENGINE')
print('==================')
print('📁 Input:', input_file.name)
print('⚡ Scale:', str(scale) + '×')
print('🧠 Model: RealESRGAN_x4plus')

for p in output_folder.iterdir():
    if p.is_file():
        p.unlink()

print('🧹 Preparing image...')

preprocessed_file = Path('/content/otaku_preprocessed.png')
img = cv2.imread(str(input_file), cv2.IMREAD_COLOR)

if img is None:
    raise Exception('❌ Could not read input image.')

noise_strength = max(1, min(15, round(TOPAZ_PROFILE['reduce_noise'] / 6)))
denoised = cv2.fastNlMeansDenoisingColored(img, None, noise_strength, noise_strength, 7, 21)
cv2.imwrite(str(preprocessed_file), denoised)

print('🚀 Running AI upscale...')

command = [
    '/content/venv311/bin/python',
    '/content/realesrgan/inference_realesrgan.py',
    '-n', 'RealESRGAN_x4plus',
    '-i', str(preprocessed_file),
    '-o', str(output_folder),
    '--outscale', str(scale),
    '--model_path', '/content/realesrgan/weights/RealESRGAN_x4plus.pth'
]

result = subprocess.run(command, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise Exception('❌ AI upscaling failed.')

ai_files = [p for p in output_folder.iterdir() if p.is_file() and p.suffix.lower() in image_extensions]

if not ai_files:
    raise Exception('❌ No AI output generated.')

ai_output = ai_files[0]
upscaled = cv2.imread(str(ai_output), cv2.IMREAD_COLOR)

if upscaled is None:
    raise Exception('❌ Could not read AI output.')

print('✨ Applying detail enhancement...')

sharpen_strength = TOPAZ_PROFILE['sharpen'] / 100.0
blurred = cv2.GaussianBlur(upscaled, (0, 0), 1.2)
enhanced = cv2.addWeighted(
    upscaled,
    1.0 + sharpen_strength * 0.45,
    blurred,
    -sharpen_strength * 0.45,
    0
)

final_output = output_folder / f'OTAKU_{input_file.stem}_x{scale}.png'
cv2.imwrite(str(final_output), enhanced)

if ai_output.exists() and ai_output != final_output:
    ai_output.unlink()

original = Image.open(input_file)
final_image = Image.open(final_output)

print()
print('🔥 OTAKU UPSCALE COMPLETE!')
print('==========================')
print('📥 Original :', original.size)
print('📤 Output   :', final_image.size)
print('📁 File     :', final_output.name)

print('\n🖼️ ORIGINAL')
display(original)

print('🖼️ OTAKU RESULT')
display(final_image)

In [ ]:
# ============================================
# 📥 OTAKU DOWNLOAD RESULT
# ============================================
from google.colab import files

output_files = [
    p for p in output_folder.iterdir()
    if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}
]

if output_files:
    output_file = output_files[0]
    print('🔥 Downloading:', output_file.name)
    files.download(str(output_file))
else:
    print('❌ No output file found.')
    print('Run the AI Upscaling Engine first.')

# 🔥 OTAKU UPSCALER V1 ROADMAP

### V1 — IMAGE UPSCALER
- ✅ Colab GPU
- ✅ Python 3.11 environment
- ✅ Real-ESRGAN AI model
- ✅ Image upload
- ✅ 2× / 4× AI upscale
- ✅ Topaz-inspired profile
- ✅ Noise reduction
- ✅ Detail enhancement
- ✅ Sharpening
- ✅ Preview
- ✅ Download

### NEXT
- ⬜ Better compression recovery
- ⬜ Dehalo
- ⬜ Deblur
- ⬜ Anime mode
- ⬜ Movie mode

### V2
- ⬜ Video upscaling
- ⬜ CC pipeline
- ⬜ 60 FPS interpolation
- ⬜ Before CC → Upscale → CC → After CC